# Step 2.5: Compare Vectors Across Personas

Core analysis: how similar are the steering vectors for the same trait when
extracted under different personas?

We compute:
- Pairwise cosine similarity between persona vectors per trait
- Magnitude ratios
- Shared vs persona-specific component decomposition
- Transfer matrices across all persona pairs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from persona_steering.config import VECTORS_DIR, Trait
from persona_steering.analysis import (
    compare_vectors,
    build_transfer_matrix,
    build_per_trait_transfer,
    decompose_shared_specific,
    compute_curvature,
)
from persona_steering.utils import load_pickle, ensure_output_dirs

ensure_output_dirs()
all_vectors = load_pickle(VECTORS_DIR / "all_vectors.pkl")
persona_slugs = list(all_vectors.keys())
print(f"Personas: {persona_slugs}")

In [ ]:
# Pick a representative layer (midpoint of extraction range)
sample_trait = list(all_vectors[persona_slugs[0]].keys())[0]
sample_layers = sorted(all_vectors[persona_slugs[0]][sample_trait].keys())
mid_layer = sample_layers[len(sample_layers) // 2]
print(f"Using layer {mid_layer} for comparison")

# Pairwise comparison per trait
traits = list(Trait)
for trait in traits:
    print(f"\n--- {trait.value} ---")
    for i, pa in enumerate(persona_slugs):
        for j, pb in enumerate(persona_slugs):
            if j <= i:
                continue
            va = all_vectors[pa][trait][mid_layer]
            vb = all_vectors[pb][trait][mid_layer]
            comp = compare_vectors(va, vb)
            print(f"  {pa} vs {pb}: cos={comp.cosine_sim:.4f}, "
                  f"mag_ratio={comp.magnitude_ratio:.4f}, "
                  f"orth_residual={comp.orthogonal_residual:.4f}")

In [ ]:
# Transfer matrix: average cosine similarity across traits
matrix = build_transfer_matrix(all_vectors, persona_slugs, traits, mid_layer)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(matrix, annot=True, fmt=".3f", xticklabels=persona_slugs,
            yticklabels=persona_slugs, cmap="RdYlBu_r", vmin=0, vmax=1, ax=ax)
ax.set_title(f"Steering Vector Transfer Matrix (Layer {mid_layer})")
plt.tight_layout()
plt.savefig("../outputs/figures/transfer_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Per-trait transfer matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, trait in zip(axes.flat, traits):
    m = build_per_trait_transfer(all_vectors, persona_slugs, trait, mid_layer)
    sns.heatmap(m, annot=True, fmt=".3f", xticklabels=persona_slugs,
                yticklabels=persona_slugs, cmap="RdYlBu_r", vmin=0, vmax=1, ax=ax)
    ax.set_title(trait.value.title())

plt.suptitle(f"Per-Trait Transfer Matrices (Layer {mid_layer})", fontsize=14)
plt.tight_layout()
plt.savefig("../outputs/figures/per_trait_transfer.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Shared vs specific decomposition per trait
for trait in traits:
    vecs = {slug: all_vectors[slug][trait][mid_layer] for slug in persona_slugs}
    decomp = decompose_shared_specific(vecs)
    print(f"\n{trait.value}: variance_explained={decomp.variance_explained:.4f}")
    for slug in persona_slugs:
        print(f"  {slug}: shared_mag={decomp.shared_magnitudes[slug]:.4f}, "
              f"specific_mag={decomp.specific_magnitudes[slug]:.4f}")

In [ ]:
# Curvature / decay analysis
curvature = compute_curvature(matrix)
print("\nCurvature analysis:")
for k, v in curvature.items():
    print(f"  {k}: {v:.4f}")